In [1]:
import sys
print("Python version:", sys.version)

Python version: 3.10.21 | packaged by Anaconda, Inc. | (main, Aug 27 2026, 14:35:39) [MSC v.1942 64 bit (AMD64)]


In [2]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA is available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("CUDA version built with PyTorch:", torch.version.cuda)
    print("GPU Device Name:", torch.cuda.get_device_name(0))
else:
    print("ระบบมองไม่เห็น GPU หรือติดตั้ง PyTorch เวอร์ชัน CPU อยู่")

PyTorch version: 2.14.1+cu130
CUDA is available: True
CUDA version built with PyTorch: 13.0
GPU Device Name: NVIDIA GeForce RTX 5050 Laptop GPU


In [3]:
%run ndjson_to_yolo.py

Acne Dataset 6 Classes.v1i.yolov8: 14468 รูป, 6 คลาส
ชื่อคลาสในไฟล์ ['blackhead', 'cyst', 'nodule', 'papule', 'pustule', 'whitehead']  ->  ใช้ ['Blackheads', 'Cysts', 'Nodules', 'Papules', 'Pustules', 'Whiteheads']
โหลดแล้วในเครื่อง 0 รูป, ต้องโหลดเพิ่ม 14468 รูป
ลิงก์รูปหมดอายุ 22/10/2026 22:00 (อีกประมาณ 163.9 ชั่วโมง)
โหลดแล้ว 14468/14468 (ล้มเหลว 0)

=== จำนวนรูปและกรอบต่อคลาส ===
split       รูป   %รูป  Blackheads       Cysts     Nodules     Papules    Pustules  Whiteheads
train     10128   70.0       11713        3319        7769       27584       19094         912
valid      2170   15.0        2313         743        1838        5661        3931         304
test       2170   15.0        2740         752        1751        6231        3950         287

=== สัดส่วน (%) ของกรอบแต่ละคลาสที่อยู่ในแต่ละชุด (ควรใกล้เคียงกับ %รูป ข้างบน) ===
train                         69.9        68.9        68.4        69.9        70.8        60.7
valid                         13.8        15.4      

In [4]:
%run clean_dataset.py static

คัดลอก D:\datasets\acne-6classes\dataset -> D:\datasets\acne-6classes-clean ...
คลาส (6): ['Blackheads', 'Cysts', 'Nodules', 'Papules', 'Pustules', 'Whiteheads']
train: 10128 รูป
valid: 2170 รูป
test: 2170 รูป

รูปซ้ำ/คล้ายกัน: 2503 กลุ่ม (ใหญ่สุด [7, 7, 7] รูป) ถูกคัดออก 3465 รูป, ซ้ำแบบเป๊ะ 62 รูป
ดูรูปเทียบคู่ได้ที่ D:\datasets\acne-6classes-clean\_dup_preview (ซ้าย = เก็บไว้, ขวา = คัดออก)
ถ้ากลุ่มใหญ่ผิดปกติ (หลายสิบรูป) ให้ลด NEAR_DUP_MAX_BITS แล้วรันใหม่

=== จำนวนกรอบต่อคลาส (หลังทำความสะอาด) ===
split       รูป  Blackheads       Cysts     Nodules     Papules    Pustules  Whiteheads
train      7007        7606        1896        6908       20504       14500         836
valid      2068        2207         699        1812        5456        3835         285
test       1866        2216         615        1676        5418        3584         283

=== สรุปปัญหาที่พบ ===
  3564  clipped_to_image             -> box clipped
  1888  near_duplicate_cross_split   -> quarantined
  1577  ne

In [5]:
import os
os.environ["PIN_MEMORY"] = "False"

from ultralytics import YOLO

clean_yaml = r"D:\datasets\acne-6classes-clean\data.yaml"

model = YOLO("yolo26s.pt")
model.train(
    data=clean_yaml,
    epochs=100,
    imgsz=640,
    batch=16,
    workers=2,
    patience=30,        # ของ YOLO เอง แทน callback เดิม
    name="new6-640",   # ตั้งชื่อรัน จะได้หาเจอง่าย
)
model.export(format="onnx", imgsz=640, simplify=True)

New https://pypi.org/project/ultralytics/8.4.177 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.169  Python-3.10.21 torch-2.14.1+cu130 CUDA:0 (NVIDIA GeForce RTX 5050 Laptop GPU, 8151MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\datasets\acne-6classes-clean\data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mo

'C:\\Users\\Ming\\runs\\detect\\new6-640\\weights\\best.onnx'

In [1]:
from ultralytics import YOLO
m = YOLO(r"C:\Users\Ming\runs\detect\new6-640\weights\best.pt")
print(m.names)

import onnxruntime as ort
s = ort.InferenceSession(r"C:\Users\Ming\runs\detect\new6-640\weights\best.onnx")
print(s.get_inputs()[0].name, s.get_inputs()[0].shape)
print(s.get_outputs()[0].name, s.get_outputs()[0].shape)

{0: 'Blackheads', 1: 'Cysts', 2: 'Nodules', 3: 'Papules', 4: 'Pustules', 5: 'Whiteheads'}
images [1, 3, 640, 640]
output0 [1, 10, 8400]
